---
title: "Parity Twine Optimizer: A Qiskit Function by ParityQC"
description: Solve optimization problems on quantum computers using Parity Twine. Minimizes gate counts and circuit depths to run quantum optimization algorithms.
---

{/* cspell:ignore parityqc QOBLIB  */}
# Parity Twine Optimizer: A Qiskit Function by ParityQC

<Admonition type="note">
  Qiskit Functions are an experimental feature available only to IBM Quantum&reg; Premium Plan, Flex Plan, and On-Prem (via IBM Quantum Platform API) Plan users. They are in preview release status and subject to change.
</Admonition>


## Overview


The Parity Twine Optimizer allows users to solve optimization problems with quantum computers using the [Parity Twine method](https://arxiv.org/abs/2501.14020).
This approach, built on the Parity Architecture, uses the ParityQC compiler to minimize the gate counts and circuit depth required to run a quantum algorithm while accounting for device connectivity.

This resource reduction enables solving industry-relevant optimization problems for sizes which are unfeasible with standard, SWAP-heavy, transpilation methods.
The Parity Twine Optimizer automates transpilation steps and can be used with minimal user input.

The Parity Twine approach outperforms all other compilation methods in terms of reducing gate counts and circuit depths for quantum algorithms such as
[the quantum approximate optimization algorithm (QAOA)](https://arxiv.org/abs/2501.14020) and [quantum Fourier transform (QFT)](https://arxiv.org/abs/2604.12465)..
By abstracting quantum information away from the physical qubits, non-local interactions are implemented without SWAP-gates. Instead, Parity Twine chains (sequences of CNOT gates) systematically distribute logical
parity information between qubits.

The optimizer uses the QAOA to minimize the energy of a given cost function. QAOA is a hybrid quantum-classical variational algorithm designed to solve combinatorial optimization problems by approximating the ground state
of a cost Hamiltonian ($H_C$). A mixing term ($H_B$) drives transitions between states to explore the solution space. An alternating sequence of these terms, controlled by parameters $\gamma$ and $\beta$, defines the unitary
$U(\gamma, \beta) = \Pi_{i=1}^p e^{-i \beta_i H_B} e^{-i \gamma_i H_C}$, which acts on the initial state.

Encoding of the problem unitary, exp($-i \beta H_C)$), requires encoding of all logical $n$-body rotation operators (exp($-i \beta J_{i,j,...n}Z_iZ_j...Z_n$)).
This can be accomplished using $n$-body generators (Parity Twine Networks), which reduce the QAOA circuit depth and two-qubit gate count.

## Description

The Parity Twine Optimizer input consists of an optimization problem, the chosen IBM Quantum hardware name, and optionally, additional arguments to tailor the transpilation and hardware execution settings. The optimizer
function routes the problem definition to the Parity Twine Compiler. A summary of the function workflow is shown schematically in Figure 1.


![Parity Twine Optimizer workflow](/docs/images/guides/parity-twine-optimizer/process_pipeline.avif "Figure 1: Summary of the Parity Twine Optimizer workflow.")

The input optimization problem is pre-processed into a (normalized) Hamiltonian. The ParityQC compiler then constructs a parameterized Parity Twine ansatz circuit, which is transpiled to a logical ansatz matching the native
gate set of the chosen hardware.

QAOA circuit parameters are classically optimized using the ["efficient depth one"](https://github.com/qiskit-community/qaoa_training_pipeline/blob/main/how_tos/efficient_depth_one.ipynb) approach and assigned to the transpiled circuit.
As one round of QAOA is used, optimal $\gamma$ and $\beta$ parameters are obtained using a grid search on a CPU without the need for costly iterative optimization
over many circuit calls. This greatly minimizes the required QPU runtime of the optimizer. Execution time per circuit is mostly determined by the constant overhead to run a job on the QPU.
With the circuit parameters set, the job is submitted to the specified IBM&reg; QPU.

Raw results are then post-processed. This step uses a local greedy search that iterates randomly through the solution string and flips bit values if this reduces the value of the cost function.

## Benchmarks

Disclaimer: Performance may depend on both the problem instance and subsequent processing steps. In some cases, classical samples and quantum-generated samples might achieve similar final solution quality
after equivalent post-processing. Evaluation should therefore consider the complete optimization workflow.
### Sherrington-Kirkpatrick model

The effectiveness of the Parity Twine Optimizer is demonstrated in Figure 2. This compares results obtained using the optimizer against random sampling for the Sherrington-Kirkpatrick model
with increasing qubits.

Post-processing of both the random and optimizer results leads to exact ground states, but the raw minimum energy output from Twine QAOA is always better than random
values. For the largest qubit numbers considered (90 and 100), hardware noise degrades the optimizer result where it reverts to raw values matching random sampling.

![Parity Twine Optimizer SK Benchmarks](/docs/images/guides/parity-twine-optimizer/benchmark_run_sk.avif "Figure 2: Benchmark results for Parity Twine Optimizer for SK model compared to random sampling. Results were obtained on
IBM Boston device with 20k shots, repeated three times.")

An introduction to the Sherrington-Kirkpatrick model and details on how to solve it with the Parity Twine Optimizer are given in the corresponding [tutorial](/docs/tutorials/parity-twine-optimizer-sk).

### Market Split problem

Specific performance metrics for the Parity Twine Optimizer applied to various instances of the Market Split (MS) problem are shown in Table 1 below.
These were obtained from the [QOBLIB - Quantum Optimization Benchmarking Library](https://github.com/ZIB-AOPT/QOBLIB) and correspond to
[submissions](https://github.com/ZIB-AOPT/QOBLIB/tree/main/01-marketsplit/submissions/20260728_Parity_Twine_Optimizer) made to the
library using the Parity Twine Optimizer.

MS violation is an accuracy metric that measures how far the solution is from satisfying all constraints, defined as $\sum_i ((Ax - b)_i)^2$. Lower values of this metric indicate more accurate solutions.
Table 1 reports averaged values for five runs of each instance using the Parity Twine Optimizer with the `ibm_boston` device as a backend and 100,000 shots.
The minimum MS violation obtained from the runs of each instance is reported.


| Instance      | Qubits | Two-qubit Gates | Two-qubit Depth | MS Violation | Mapping Time (s) | QPU Runtime (s) | Post-process Time (s) |
|:--------------|:-------|:----------------|:----------------|:-------------|:-----------------|:----------------|:----------------------|
| ms_03_050_002 | 20     | 399             | 76              | 0          | 45               | 29              | 71                    |
| ms_03_100_022 | 20     | 399             | 76              | 0          | 30               | 29              | 77                    |
| ms_03_200_177 | 20     | 399             | 76              | 0          | 33               | 29              | 63                    |
| ms_04_050_001 | 30     | 899             | 116             | 1          | 68               | 29              | 216                   |
| ms_04_100_003 | 30     | 899             | 116             | 2          | 89               | 29              | 187                   |
| ms_04_200_030 | 30     | 899             | 116             | 3          | 89               | 29              | 219                   |
| ms_05_100_003 | 40     | 1599            | 156             | 3          | 237              | 30              | 364                   |
| ms_06_050_001 | 50     | 2499            | 196             | 4          | 499              | 30              | 464                   |
| ms_07_050_001 | 60     | 3599            | 236             | 17         | 858              | 31              | 637                   |
| ms_08_050_001 | 70     | 4899            | 276             | 36         | 1561             | 31              | 773                   |

Table 1: Performance metrics for different instances of the Market Split problem. Instance names are for corresponding examples in the [QOBLIB](https://github.com/ZIB-AOPT/QOBLIB) problem library.
         As mentioned in the description section, the QPU time is minimal and almost constant as only one round of QAOA is used with optimal circuit parameters obtained on a CPU.


An introduction to the Market Split problem and details on how to solve it with the Parity Twine Optimizer are given in the corresponding [tutorial](/docs/tutorials/parity-twine-optimizer-ms).

### Maximum Independent Set problem

Figure 3 compares results obtained using the Parity Twine optimizer against random sampling for the maximum independent set problem for selected instances
obtained from the [QOBLIB - Quantum Optimization Benchmarking Library](https://github.com/ZIB-AOPT/QOBLIB). These range from 18 (mammalia-kangaroo-interactions) to
114 (es60fst03) qubits.

For small instances, the raw Maximum Independent Set size results form the optimizer and random sampling are similar but for larger instances, the optimizer gives far better results.
Post-processing of both the random sampled and optimizer results again leads to set sizes in agreement with exact values.

![Parity Twine Optimizer MIS Benchmarks](/docs/images/guides/parity-twine-optimizer/benchmark_run_mis.avif "Figure 3: Benchmark results for Parity Twine Optimizer for MIS model compared to random sampling.
Results were obtained on IBM Boston device with 20K shots, repeated three times.")

## Get started

Authenticate using your [IBM Quantum Platform API token](/docs/guides/functions-get-started) and select the Qiskit Function as:

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Authenticate and load the Qiskit Function
catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
)
function = catalog.load("parityqc/parity-twine-optimizer")

## Example

The Parity Twine Optimizer can be applied to a simple QUBO example as follows. The problem is first defined using a dictionary:

In [ ]:
problem = {"()": 3, "(0,)": 1, "(0, 1)": 2, "(1, 2)": -1}

Then the variable type has be defined as `binary` or `spin`:

In [ ]:
variable_type = "spin"

This defines the objective function $3 + s_0 + 2 s_0s_1 - s_1s_2$ where $s_i \in \{-1, 1\}$.

A backend should be chosen, for example:

In [ ]:
backend_name = "ibm_phoenix"

Use `catalog.backends()` to see a list of available backends from the user account/instance.

If no backend is specified, the least-busy backend (the one with the least number of pending jobs) is automatically selected

With the optimizer loaded (see previous), it can then be run for the problem using default options via:

In [ ]:
function_job = function.run(
    problem=problem, variable_type=variable_type, backend_name=backend_name
)
print(f"Job ID: {function_job.job_id}")

The status of the job can be checked via:

In [ ]:
# Monitor the job status
function_job.status()

Results are retrieved using:

In [ ]:
# Retrieve the job result if the status is DONE
result = function_job.result()

result

Printing the results gives:
```
{
    'solution': {'0': -1, '1': 1, '2': 1},
    'objective_value': -1.0,
    'solution_bitstring': '100',
    'metadata': {
        'circuit_metrics': {
            'depth': 20,
            'gate_count': 198,
            'two_qubit_gate_depth': 4,
            'two_qubit_gate_count': 4,
            'num_qubits': 3,
            'operations': {'delay': 158, 'rz': 17, 'sx': 14, 'cz': 4, 'measure': 3, 'x': 2}
        },
        'solver_info': {
            'variable_mapping': {'0': 0, '1': 1, '2': 2},
            'bitstring_distributions': {
                'before_postprocessing': {'100': 71760, ... },
                'after_postprocessing': {'100': 88440, '011': 11560}
            },
            'best_parameters': {
                'beta': [-0.46259546391008877],
                'gamma': [0.6181957189727373]
            }
        },
        'resource_usage': {
            'RUNNING: MAPPING': {'CPU_TIME': 42.878},
            'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.094},
            'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 790.623},
            'RUNNING: EXECUTING_QPU': {'QPU_TIME': 28.0},
            'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.828}
        }
    }
}
```
The `solution` dictionary corresponds to the qubits defined in the problem ($s_0, s_1$, and $s_2$) and gives their optimized spin values.

The `objective_value` is the cost or energy of the optimization problem. It quantifies the `solution` quality.

`metadata` gives information on the transpilation (two-qubit gate counts/depth, gates used, active qubits) and various runtimes for both quantum and classical compute time.

Additionally, `metadata` provides a glimpse into the internal workings of the algorithm. You can see the optimized variational parameters (beta and gamma) and review how the distribution of measured bitstrings improved after classical post-processing was applied to the raw quantum results.

All bitstrings in the result output can be mapped to the problem variables using the `variable_mapping`.
For example, you can map the `'solution_bitstring': '100'` with the `'variable_mapping': {'0': 0, '1': 1, '2': 2}` to the binary solution: `{'0': 1, '1': 0, '2': 0}`.
Since you are solving the problem in terms of spin variables, you need to apply the binary to spin transformation  (`0` -> `1`, `1` -> `-1` ).
You then arrive at the final solution `'solution': {'0': -1, '1': 1, '2': 1}`.

Full details of the input to and output from the Parity Twine Optimizer function can be found in the [API reference](/docs/api/functions/parity-twine-optimizer-api).

## Changelog

### [1.0.1] - 2026-09-15
**Notes**

This release marks the first stable production version of the Parity Twine Optimizer Qiskit Function. Future releases are expected to introduce
- higher-order problem compilation,
- additional optimization algorithms, and
- improvements in pre- and post-processing.


**Features**
- Solve sparse and dense QUBO problems with the efficient Parity Twine Compiler

**Documentation**
- Getting started user guide.
- Tutorials with examples for Market-Split and Sherrington-Kirkpatrick problems.
- API documentation.

## Get support

Contact [ParityQC](mailto:support@parityqc.com) for any questions or issues

## Next steps

- Request access to the function by completing this [form](https://parityqc.com/products/parity-twine-optimizer/free-trial).
- Visit the [API reference](/docs/api/functions/parity-twine-optimizer-api) for this Qiskit Function.
- Try the [tutorial](/docs/tutorials/parity-twine-optimizer-ms) for applying the Parity Twine Optimizer to the Market-Split problem.
- Try the [tutorial](/docs/tutorials/parity-twine-optimizer-sk) for applying the Parity Twine Optimizer to the Sherrington-Kirkpatrick model.
- Review the [Connectivity-aware Synthesis of Quantum Algorithms, Drier et al. (2025)](https://arxiv.org/abs/2501.14020) ArXiv preprint.